Étape 5 — Modélisation prédictive de la performance finale

Objectif :Prédire la performance finale des apprenants à partir du profil + test initial + recommandation.
On doit éviter la fuite de données.
Donc on ne met pas dans X :
score_final, niveau_final, progression, reussite_finale.

# Cellule 1 — Imports

In [19]:
# ============================================================
# ÉTAPE 5 — MODÉLISATION PRÉDICTIVE DE LA PERFORMANCE FINALE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

# Cellule 2 — Chargement des données

In [20]:
# ============================================================
# 5.1. Chargement du dataset de prédiction
# ============================================================

INPUT_DIR = Path("outputs/02_scoring_pedagogique")
INPUT_RECO_DIR = Path("outputs/04_recommandation_parcours")
OUTPUT_DIR = Path("outputs/05_modelisation_predictive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_DATASET_PREDICTION = INPUT_DIR / "05_dataset_prediction_score.csv"
PATH_RECOMMANDATIONS = INPUT_RECO_DIR / "01_recommandations_initiales.csv"

df_prediction = pd.read_csv(PATH_DATASET_PREDICTION)
df_recommandations = pd.read_csv(PATH_RECOMMANDATIONS)

print("Dataset prédiction :", df_prediction.shape)
print("Recommandations :", df_recommandations.shape)

display(df_prediction.head())

Dataset prédiction : (128, 91)
Recommandations : (305, 64)


,ins_CreatedAt,IDENTIFICATION,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,init_CreatedAt,init_Date du test,init_Q1 - Resume ventes Excel,init_Q2 - Import CSV Excel,init_Q3 - Modele donnees Excel,init_Q4 - Mauvaise pratique visu Excel,init_Q5 - 2e grande valeur Excel,init_Q6 - Mesure vs Colonne DAX,init_Q7 - CA annee precedente DAX,init_Q8 - Vue Modele Power BI,init_Q9 - Acces directeurs regionaux,init_Q10 - 12 commerciaux 3 indicateurs,init_Q11 - Bibliotheque CSV Python,init_Q12 - Overfitting Underfitting,init_Q13 - Segmentation 50000 clients,init_Q14 - Deployer modele Python API,init_Q15 - Valeurs manquantes 30pc,init_Q16 - Role system prompt LLM,init_Q17 - Assistant IA PDF financiers,init_Q18 - Role embedding dans RAG,init_Q19 - Agent IA selection outil,init_Q20 - Sortie fiable LLM tableau,final_RowId,final_CreatedAt,final_InscriptionId,final_Filiere,final_Niveau d'Etude,final_Q01,final_Q02,final_Q03,final_Q04,final_Q05,final_Q06,final_Q07,final_Q08,final_Q09,final_Q10,final_Q11,final_Q12,final_Q13,final_Q14,final_Q15,final_parcours_test_final,final_feuille_source_finale,score_initial_DA,score_initial_DA_pct,niveau_initial_DA,score_initial_BI,score_initial_BI_pct,niveau_initial_BI,score_initial_DS,score_initial_DS_pct,niveau_initial_DS,score_initial_IA,score_initial_IA_pct,niveau_initial_IA,score_initial_total,score_initial_pourcentage,niveau_initial_global,unknown_total_initial,parcours_test_final,score_final_total,score_final_pourcentage,niveau_final,unknown_total_final,progression_absolue_pct,reussite_finale,progression_absolue,cluster_initial,cluster_final,statut_progression
0,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,2026-04-03 13:28:48+00:00,2026-04-03 13:28:47+00:00,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,15,2026-06-04 20:03:57+00:00,3,Droit / Sciences politiques,Master 2 (M2),"B) Extraire le fichier, nettoyer/transformer l...",A) Les mesures et cles vers les dimensions,B) Workspace / Espace de travail > Skills,A) Administration > Reglages > Integrations,C) Administration > Fonctions > Nouvelle fonction,D) Valider et structurer les donnees recues et...,D) Generer des graphiques interactifs ou expor...,C) Stocker et exposer par API les contenus/rap...,C) Les prompts temporaires non publies,C) Afficher les rapports et declencher/recuper...,C) Des nodes connectes qui executent des actio...,"B) Lancer l'ETL, appeler les API, publier dans...",A) Piloter ou modifier des workflows et integr...,"D) Verifier l'endpoint/API appele, le slug/id ...",D) Administration > Reglages > Connexion,IA,FINAL-TEST-IA,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,20,IA,14,93.333333,Avancé,0,93.333333,1,93.333333,Débutant,Avancé,Progression
1,2026-04-01 06:30:52+00:00,DIEG2026-007,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Debutant,Aucun,Intermediaire,Debutant,"[""Acquerir des competences techniques"", "" Real...",Developpeur Python / IA,"[""En semaine - Soir"", "" Week-end""]",WhatsApp,2002,2002.0,24.0,2026-04-01 06:41:29+00:00,2026-04-01 06:41:26+00:00,B. Tableau croise dynamique (TCD),C. Formules SI imbriquees,B. TCD simple,E. Je ne sais pas,B. GRANDE

# Cellule 3 — Fusion avec recommandations

In [21]:
# ============================================================
# 5.2. Ajout des variables de recommandation
# ============================================================

colonnes_reco_utiles = [
    "IDENTIFICATION",
    "parcours_recommande",
    "domaine_plus_faible",
    "lacunes_detectees"
]

df_model = df_prediction.merge(
    df_recommandations[colonnes_reco_utiles],
    on="IDENTIFICATION",
    how="left",
    validate="one_to_one"
)

print("Dataset modèle :", df_model.shape)

print("\nCibles disponibles :")
print("score_final_pourcentage :", "score_final_pourcentage" in df_model.columns)
print("reussite_finale :", "reussite_finale" in df_model.columns)
print("niveau_final :", "niveau_final" in df_model.columns)

display(df_model.head())

Dataset modèle : (128, 94)

Cibles disponibles :
score_final_pourcentage : True
reussite_finale : True
niveau_final : True


,ins_CreatedAt,IDENTIFICATION,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,init_CreatedAt,init_Date du test,init_Q1 - Resume ventes Excel,init_Q2 - Import CSV Excel,init_Q3 - Modele donnees Excel,init_Q4 - Mauvaise pratique visu Excel,init_Q5 - 2e grande valeur Excel,init_Q6 - Mesure vs Colonne DAX,init_Q7 - CA annee precedente DAX,init_Q8 - Vue Modele Power BI,init_Q9 - Acces directeurs regionaux,init_Q10 - 12 commerciaux 3 indicateurs,init_Q11 - Bibliotheque CSV Python,init_Q12 - Overfitting Underfitting,init_Q13 - Segmentation 50000 clients,init_Q14 - Deployer modele Python API,init_Q15 - Valeurs manquantes 30pc,init_Q16 - Role system prompt LLM,init_Q17 - Assistant IA PDF financiers,init_Q18 - Role embedding dans RAG,init_Q19 - Agent IA selection outil,init_Q20 - Sortie fiable LLM tableau,final_RowId,final_CreatedAt,final_InscriptionId,final_Filiere,final_Niveau d'Etude,final_Q01,final_Q02,final_Q03,final_Q04,final_Q05,final_Q06,final_Q07,final_Q08,final_Q09,final_Q10,final_Q11,final_Q12,final_Q13,final_Q14,final_Q15,final_parcours_test_final,final_feuille_source_finale,score_initial_DA,score_initial_DA_pct,niveau_initial_DA,score_initial_BI,score_initial_BI_pct,niveau_initial_BI,score_initial_DS,score_initial_DS_pct,niveau_initial_DS,score_initial_IA,score_initial_IA_pct,niveau_initial_IA,score_initial_total,score_initial_pourcentage,niveau_initial_global,unknown_total_initial,parcours_test_final,score_final_total,score_final_pourcentage,niveau_final,unknown_total_final,progression_absolue_pct,reussite_finale,progression_absolue,cluster_initial,cluster_final,statut_progression,parcours_recommande,domaine_plus_faible,lacunes_detectees
0,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,2026-04-03 13:28:48+00:00,2026-04-03 13:28:47+00:00,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,15,2026-06-04 20:03:57+00:00,3,Droit / Sciences politiques,Master 2 (M2),"B) Extraire le fichier, nettoyer/transformer l...",A) Les mesures et cles vers les dimensions,B) Workspace / Espace de travail > Skills,A) Administration > Reglages > Integrations,C) Administration > Fonctions > Nouvelle fonction,D) Valider et structurer les donnees recues et...,D) Generer des graphiques interactifs ou expor...,C) Stocker et exposer par API les contenus/rap...,C) Les prompts temporaires non publies,C) Afficher les rapports et declencher/recuper...,C) Des nodes connectes qui executent des actio...,"B) Lancer l'ETL, appeler les API, publier dans...",A) Piloter ou modifier des workflows et integr...,"D) Verifier l'endpoint/API appele, le slug/id ...",D) Administration > Reglages > Connexion,IA,FINAL-TEST-IA,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,20,IA,14,93.333333,Avancé,0,93.333333,1,93.333333,Débutant,Avancé,Progression,DA,DA,"DA, BI, DS, IA"
1,2026-04-01 06:30:52+00:00,DIEG2026-007,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Debutant,Aucun,Intermediaire,Debutant,"[""Acquerir des competences techniques"", "" Real...",Developpeur Python / IA,"[""En semaine - Soir"", "" Week-end""]",WhatsApp,2002,2002.0,24.0,2026-04-01 06:41:29+00:00,2026-04-01 06:41:26+00:00,B. Tableau croise dy

# Corrections appliquées:
1.Suppression des 20 réponses brutes du test initial (init_Q1 à init_Q20) — redondantes avec les scores déjà agrégés (score_initial_DA_pct, BI_pct, DS_pct, IA_pct, pourcentage)
2.Suppression de 4 colonnes de dates/horodatages (CreatedAt, UpdatedAt, Date du test) — traitées à tort comme catégorielles
3.Suppression de 3 colonnes dérivées redondantes (parcours_recommande, domaine_plus_faible, lacunes_detectees) — fonctions déterministes des scores déjà présents
4.Passage d'un split train/test unique à une validation croisée K-Fold (k=5) sur l'ensemble du dataset, au lieu de X_train/X_test séparés
5.Régularisation des modèles à arbres :
-DecisionTree : max_depth=4, min_samples_leaf=8 (avant : max_depth=5 seul)
-RandomForest : max_depth=4, min_samples_leaf=8, max_features="sqrt" (avant : max_depth=None)
-GradientBoosting : max_depth=2 (avant : max_depth=3)
6.Ajout du suivi du surapprentissage (return_train_score=True + calcul de ecart_R2_train_test) — bonne pratique ajoutée spontanément, pas demandée initialement

# Cellule 4 — Sélection propre des variables X

In [23]:
# ============================================================
# 5.3. Définition des variables explicatives et des cibles
# VERSION CORRIGÉE : suppression redondance, bruit et fuite
# ============================================================

# Cibles
y_regression = df_model["score_final_pourcentage"]
y_classification = df_model["reussite_finale"]
y_niveau = df_model["niveau_final"]

# ------------------------------------------------------------
# 1. Colonnes interdites : fuite de données liée au test final
# ------------------------------------------------------------

colonnes_fuite = [
    col for col in df_model.columns
    if (
        col.startswith("final_")
        or col.startswith("score_final")
        or col.startswith("unknown_final")
        or col in [
            "score_final_total",
            "score_final_pourcentage",
            "niveau_final",
            "cluster_final",
            "reussite_finale",
            "progression_absolue",
            "progression_absolue_pct",
            "progression_initial_finale_pct",
            "statut_progression",
            "parcours_test_final",
            "feuille_source_finale"
        ]
    )
]

# ------------------------------------------------------------
# 2. Colonnes brutes du test initial à supprimer
#    Elles sont déjà résumées par :
#    score_initial_DA_pct, score_initial_BI_pct,
#    score_initial_DS_pct, score_initial_IA_pct,
#    score_initial_pourcentage
# ------------------------------------------------------------

colonnes_reponses_brutes_initial = [
    col for col in df_model.columns
    if (
        col.lower().startswith("init_q")
        or col.lower().startswith("initial_q")
    )
]

# ------------------------------------------------------------
# 3. Colonnes de dates / horodatages à supprimer
#    Elles ne doivent pas passer dans OneHotEncoder
# ------------------------------------------------------------

colonnes_dates_bruit = [
    col for col in df_model.columns
    if any(mot in col.lower() for mot in [
        "createdat",
        "updatedat",
        "date du test",
        "date_test",
        "timestamp",
        "horodatage"
    ])
]

# ------------------------------------------------------------
# 4. Colonnes redondantes déterministes des scores initiaux
# ------------------------------------------------------------

colonnes_recommandation_deterministes = [
    col for col in [
        "parcours_recommande",
        "domaine_plus_faible",
        "lacunes_detectees"
    ]
    if col in df_model.columns
]

# ------------------------------------------------------------
# 5. Colonnes techniques
# ------------------------------------------------------------

colonnes_techniques = [
    col for col in [
        "IDENTIFICATION"
    ]
    if col in df_model.columns
]

# ------------------------------------------------------------
# 6. Liste finale des colonnes exclues
# ------------------------------------------------------------

colonnes_exclues = sorted(list(set(
    colonnes_fuite
    + colonnes_reponses_brutes_initial
    + colonnes_dates_bruit
    + colonnes_recommandation_deterministes
    + colonnes_techniques
)))

X = df_model.drop(columns=colonnes_exclues)

# On garde uniquement les types exploitables
X = X.select_dtypes(include=["int64", "float64", "object", "bool"]).copy()

print("Nombre de variables explicatives :", X.shape[1])
print("Nombre d'apprenants :", X.shape[0])

print("\nColonnes exclues :", len(colonnes_exclues))
print("Réponses brutes initiales exclues :", len(colonnes_reponses_brutes_initial))
print("Colonnes de dates exclues :", len(colonnes_dates_bruit))
print("Colonnes déterministes exclues :", colonnes_recommandation_deterministes)

print("\nVérification des colonnes restantes contenant init_Q :")
print([col for col in X.columns if col.lower().startswith("init_q")])

print("\nVérification des colonnes restantes contenant CreatedAt/date :")
print([col for col in X.columns if "createdat" in col.lower() or "date" in col.lower()])

display(X.head())

Nombre de variables explicatives : 36
Nombre d'apprenants : 128

Colonnes exclues : 58
Réponses brutes initiales exclues : 20
Colonnes de dates exclues : 4
Colonnes déterministes exclues : ['parcours_recommande', 'domaine_plus_faible', 'lacunes_detectees']

Vérification des colonnes restantes contenant init_Q :
[]

Vérification des colonnes restantes contenant CreatedAt/date :
[]


,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,score_initial_DA,score_initial_DA_pct,niveau_initial_DA,score_initial_BI,score_initial_BI_pct,niveau_initial_BI,score_initial_DS,score_initial_DS_pct,niveau_initial_DS,score_initial_IA,score_initial_IA_pct,niveau_initial_IA,score_initial_total,score_initial_pourcentage,niveau_initial_global,unknown_total_initial,unknown_total_final,cluster_initial
0,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,20,0,Débutant
1,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Debutant,Aucun,Intermediaire,Debutant,"[""Acquerir des competences techniques"", "" Real...",Developpeur Python / IA,"[""En semaine - Soir"", "" Week-end""]",WhatsApp,2002,2002.0,24.0,2,40.0,Débutant,1,20.0,Débutant,4,80.0,Avancé,1,20.0,Débutant,8,40.0,Débutant,6,0,Débutant
2,2003,Masculin,Diana,"EGS - Economie, Gestion & Sociologie",Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Intermediaire,Debutant,Intermediaire,Debutant,"[""Acquerir des competences techniques"", "" Real...",Data Analyst,"[""En semaine - Apres-midi""]",Facebook,2003,2003.0,23.0,1,20.0,Débutant,0,0.0,Débutant,1,20.0,Débutant,2,40.0,Débutant,4,20.0,Débutant,0,0,Débutant
3,2000,Feminin,Fitovinany,"EGS - Economie, Gestion & Sociologie",Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Intermediaire,Debutant,Intermediaire,Debutant,"[""Curiosite"", "" Acquerir des competences techn...",Data Analyst,"[""En semaine - Apres-midi""]",Facebook,2000,2000.0,26.0,3,60.0,Intermédiaire,2,40.0,Débutant,1,20.0,Débutant,1,20.0,Débutant,7,35.0,Débutant,0,1,Débutant
4,2001,Feminin,Diana,IST - Informatique & Sciences des Technologies,Licence 2 (L2),Diplome(e) en recherche d emploi,Intermediaire,Intermediaire,Aucun,Debutant,Intermediaire,"[""Acquerir des competences techniques"", "" Amel...",Entrepreneur,"[""Flexible""]",Ami(e),2001,2001.0,25.0,0,0.0,Débutant,0,0.0,Débutant,1,20.0,Débutant,1,20.0,Débutant,2,10.0,Débutant,12,1,Débutant


In [24]:
cardinalites = X[colonnes_categorielles].nunique().sort_values(ascending=False)
print(cardinalites)
print("\nColonnes à risque (cardinalité > 10) :")
print(cardinalites[cardinalites > 10])

ins_Motivation             63
ins_Disponibilité          28
ins_Filière                11
ins_Région d'origine       11
ins_Objectif pro            7
ins_Niveau d'étude          6
ins_Statut actuel           6
ins_Source info             6
ins_Niveau informatique     4
ins_Niveau Excel            4
ins_Niveau Power BI         4
ins_Niveau Python           4
ins_Niveau IA               4
niveau_initial_DS           3
niveau_initial_global       3
niveau_initial_IA           3
cluster_initial             3
niveau_initial_BI           3
niveau_initial_DA           3
ins_Genre                   2
dtype: int64

Colonnes à risque (cardinalité > 10) :
ins_Motivation          63
ins_Disponibilité       28
ins_Filière             11
ins_Région d'origine    11
dtype: int64


# Cellule 4 bis — Réduction des catégories à risque

In [25]:
# ============================================================
# 5.3 bis. Réduction des variables catégorielles à forte cardinalité
# ============================================================

# Colonnes textuelles libres à supprimer
colonnes_texte_libre_a_supprimer = [
    col for col in [
        "ins_Motivation",
        "ins_Disponibilité"
    ]
    if col in X.columns
]

X = X.drop(columns=colonnes_texte_libre_a_supprimer)

print("Colonnes textuelles libres supprimées :")
print(colonnes_texte_libre_a_supprimer)


# Colonnes catégorielles à regrouper
colonnes_a_regrouper = [
    col for col in [
        "ins_Filière",
        "ins_Région d'origine"
    ]
    if col in X.columns
]


def regrouper_modalites_rares(serie, min_effectif=5):
    """
    Regroupe les modalités rares dans la catégorie 'Autre'.
    Une modalité est considérée rare si son effectif est < min_effectif.
    """
    effectifs = serie.value_counts(dropna=False)
    modalites_frequentes = effectifs[effectifs >= min_effectif].index
    
    return serie.where(serie.isin(modalites_frequentes), "Autre")


for col in colonnes_a_regrouper:
    X[col] = regrouper_modalites_rares(X[col], min_effectif=5)

print("\nColonnes regroupées :")
print(colonnes_a_regrouper)

print("\nNouvelle taille de X :", X.shape)

print("\nNouvelle cardinalité des variables catégorielles :")
colonnes_categorielles_temp = X.select_dtypes(include=["object"]).columns.tolist()
display(X[colonnes_categorielles_temp].nunique().sort_values(ascending=False))

Colonnes textuelles libres supprimées :
['ins_Motivation', 'ins_Disponibilité']

Colonnes regroupées :
['ins_Filière', "ins_Région d'origine"]

Nouvelle taille de X : (128, 34)

Nouvelle cardinalité des variables catégorielles :


ins_Filière                7
ins_Objectif pro           7
ins_Niveau d'étude         6
ins_Statut actuel          6
ins_Source info            6
ins_Région d'origine       5
ins_Niveau IA              4
ins_Niveau informatique    4
ins_Niveau Excel           4
ins_Niveau Power BI        4
ins_Niveau Python          4
niveau_initial_DS          3
niveau_initial_global      3
niveau_initial_IA          3
cluster_initial            3
niveau_initial_BI          3
niveau_initial_DA          3
ins_Genre                  2
dtype: int64

In [28]:
# ============================================================
# Liste des variables explicatives restantes
# ============================================================

print("Nombre total de variables explicatives :", X.shape[1])

liste_variables = X.columns.tolist()

for i, col in enumerate(liste_variables, start=1):
    print(f"{i}. {col}")

Nombre total de variables explicatives : 34
1. ins_Année de naissance
2. ins_Genre
3. ins_Région d'origine
4. ins_Filière
5. ins_Niveau d'étude
6. ins_Statut actuel
7. ins_Niveau informatique
8. ins_Niveau Excel
9. ins_Niveau Power BI
10. ins_Niveau Python
11. ins_Niveau IA
12. ins_Objectif pro
13. ins_Source info
14. ins_annee_naissance_originale
15. ins_annee_naissance
16. ins_age
17. score_initial_DA
18. score_initial_DA_pct
19. niveau_initial_DA
20. score_initial_BI
21. score_initial_BI_pct
22. niveau_initial_BI
23. score_initial_DS
24. score_initial_DS_pct
25. niveau_initial_DS
26. score_initial_IA
27. score_initial_IA_pct
28. niveau_initial_IA
29. score_initial_total
30. score_initial_pourcentage
31. niveau_initial_global
32. unknown_total_initial
33. unknown_total_final
34. cluster_initial


# Cellule 5  — Prétraitement après nettoyage

In [26]:
# ============================================================
# 5.4. Prétraitement des variables après nettoyage
# VERSION CORRIGÉE
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

colonnes_numeriques = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
colonnes_categorielles = X.select_dtypes(include=["object"]).columns.tolist()

print("Variables numériques :", len(colonnes_numeriques))
print("Variables catégorielles :", len(colonnes_categorielles))

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), colonnes_numeriques),

        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), colonnes_categorielles)
    ]
)

print("Prétraitement défini.")

Variables numériques : 16
Variables catégorielles : 18
Prétraitement défini.


In [29]:
# ============================================================
# Variables numériques et catégorielles
# ============================================================

colonnes_numeriques = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
colonnes_categorielles = X.select_dtypes(include=["object"]).columns.tolist()

print("Variables numériques :", len(colonnes_numeriques))
for i, col in enumerate(colonnes_numeriques, start=1):
    print(f"{i}. {col}")

print("\nVariables catégorielles :", len(colonnes_categorielles))
for i, col in enumerate(colonnes_categorielles, start=1):
    print(f"{i}. {col}")

Variables numériques : 16
1. ins_Année de naissance
2. ins_annee_naissance_originale
3. ins_annee_naissance
4. ins_age
5. score_initial_DA
6. score_initial_DA_pct
7. score_initial_BI
8. score_initial_BI_pct
9. score_initial_DS
10. score_initial_DS_pct
11. score_initial_IA
12. score_initial_IA_pct
13. score_initial_total
14. score_initial_pourcentage
15. unknown_total_initial
16. unknown_total_final

Variables catégorielles : 18
1. ins_Genre
2. ins_Région d'origine
3. ins_Filière
4. ins_Niveau d'étude
5. ins_Statut actuel
6. ins_Niveau informatique
7. ins_Niveau Excel
8. ins_Niveau Power BI
9. ins_Niveau Python
10. ins_Niveau IA
11. ins_Objectif pro
12. ins_Source info
13. niveau_initial_DA
14. niveau_initial_BI
15. niveau_initial_DS
16. niveau_initial_IA
17. niveau_initial_global
18. cluster_initial


In [30]:
df_liste_variables = pd.DataFrame({
    "variable": X.columns,
    "type": X.dtypes.astype(str).values
})

PATH_LISTE_VARIABLES = OUTPUT_DIR / "liste_variables_modelisation.csv"

df_liste_variables.to_csv(
    PATH_LISTE_VARIABLES,
    index=False,
    encoding="utf-8-sig"
)

display(df_liste_variables)

print("Fichier exporté :", PATH_LISTE_VARIABLES)

,variable,type
0,ins_Année de naissance,int64
1,ins_Genre,object
2,ins_Région d'origine,object
3,ins_Filière,object
4,ins_Niveau d'étude,object
5,ins_Statut actuel,object
6,ins_Niveau informatique,object
7,ins_Niveau Excel,object
8,ins_Niveau Power BI,object
9,ins_Niveau Python,object


Fichier exporté : outputs/05_modelisation_predictive/liste_variables_modelisation.csv


Je garderais une version plus propre comme ceci :

ins_age
ins_Genre
ins_Région d'origine
ins_Filière
ins_Niveau d'étude
ins_Statut actuel
ins_Niveau informatique
ins_Niveau Excel
ins_Niveau Power BI
ins_Niveau Python
ins_Niveau IA
ins_Objectif pro
ins_Source info

score_initial_DA_pct
score_initial_BI_pct
score_initial_DS_pct
score_initial_IA_pct
score_initial_pourcentage
unknown_total_initial
niveau_initial_global

# Code pour nettoyer

In [ ]:
colonnes_a_supprimer = [
    "unknown_total_final",

    "ins_Année de naissance",
    "ins_annee_naissance_originale",
    "ins_annee_naissance",

    "score_initial_DA",
    "score_initial_BI",
    "score_initial_DS",
    "score_initial_IA",
    "score_initial_total",

    "niveau_initial_DA",
    "niveau_initial_BI",
    "niveau_initial_DS",
    "niveau_initial_IA",

    "cluster_initial"
]

colonnes_a_supprimer = [
    col for col in colonnes_a_supprimer
    if col in X.columns
]

X = X.drop(columns=colonnes_a_supprimer)

print("Nouvelle taille de X :", X.shape)
print("Colonnes supprimées :", colonnes_a_supprimer)

display(pd.DataFrame({
    "variable": X.columns,
    "type": X.dtypes.astype(str).values
}))

# Cellule 7 corrigée — Modélisation avec validation croisée K-Fold

In [27]:
# ============================================================
# 5.6. Modélisation régression avec validation croisée K-Fold
# Objectif : corriger l'instabilité du train_test_split sur 128 apprenants
# ============================================================

from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

modeles_regression = {
    "DummyRegressor": DummyRegressor(strategy="mean"),

    "Ridge": Ridge(
        alpha=10.0,
        random_state=42
    ),

    "DecisionTree": DecisionTreeRegressor(
        random_state=42,
        max_depth=4,
        min_samples_leaf=8
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        max_depth=4,
        min_samples_leaf=8,
        max_features="sqrt"
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2
    )
}

scoring = {
    "R2": "r2",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error"
}

resultats_regression = []

for nom_modele, modele in modeles_regression.items():

    pipeline = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", modele)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y_regression,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    resultats_regression.append({
        "modele": nom_modele,

        "MAE_moyen": round(-scores["test_MAE"].mean(), 4),
        "MAE_ecart_type": round(scores["test_MAE"].std(), 4),

        "RMSE_moyen": round(-scores["test_RMSE"].mean(), 4),
        "RMSE_ecart_type": round(scores["test_RMSE"].std(), 4),

        "R2_moyen": round(scores["test_R2"].mean(), 4),
        "R2_ecart_type": round(scores["test_R2"].std(), 4),

        "R2_train_moyen": round(scores["train_R2"].mean(), 4),
        "R2_train_ecart_type": round(scores["train_R2"].std(), 4),

        "ecart_R2_train_test": round(
            scores["train_R2"].mean() - scores["test_R2"].mean(),
            4
        )
    })

df_resultats_regression = pd.DataFrame(resultats_regression)

df_resultats_regression = df_resultats_regression.sort_values(
    by="RMSE_moyen",
    ascending=True
).reset_index(drop=True)

print("===== Résultats des modèles de régression avec validation croisée =====")
display(df_resultats_regression)

meilleur_modele_regression_nom = df_resultats_regression.iloc[0]["modele"]

print("\nMeilleur modèle selon le RMSE moyen :", meilleur_modele_regression_nom)

PATH_RESULTATS_REGRESSION = OUTPUT_DIR / "01_resultats_modeles_regression_cv_corrige.csv"

df_resultats_regression.to_csv(
    PATH_RESULTATS_REGRESSION,
    index=False,
    encoding="utf-8-sig"
)

print("Fichier exporté :", PATH_RESULTATS_REGRESSION)

===== Résultats des modèles de régression avec validation croisée =====


,modele,MAE_moyen,MAE_ecart_type,RMSE_moyen,RMSE_ecart_type,R2_moyen,R2_ecart_type,R2_train_moyen,R2_train_ecart_type,ecart_R2_train_test
0,RandomForest,20.0294,1.5136,23.1104,1.5677,1.510000e-02,2.430000e-02,0.1879,0.0147,1.728000e-01
1,DummyRegressor,20.5783,1.6827,23.5198,1.7576,-1.900000e-02,1.790000e-02,0.0000,0.0000,1.900000e-02
2,GradientBoosting,19.9883,2.1104,24.5783,1.8911,-1.158000e-01,8.970000e-02,0.6397,0.0285,7.556000e-01
3,DecisionTree,21.7986,1.8402,26.6399,2.1951,-3.138000e-01,1.543000e-01,0.2864,0.0495,6.002000e-01
4,Ridge,3839.7909,7638.7558,19497.1108,38946.2534,-3.501101e+06,7.002201e+06,0.4274,0.0151,3.501101e+06



Meilleur modèle selon le RMSE moyen : RandomForest
Fichier exporté : outputs/05_modelisation_predictive/01_resultats_modeles_regression_cv_corrige.csv


In [16]:
# ============================================================
# 5.6. Modélisation régression avec validation croisée K-Fold
# VERSION CORRIGÉE : anti-explosion Ridge + anti-surapprentissage GB
# ============================================================

from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import RidgeCV

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

modeles_regression = {
    "DummyRegressor": DummyRegressor(strategy="mean"),

    # CORRECTIF 1 : RidgeCV avec grille d'alpha au lieu d'une valeur fixe
    # (alpha=10 était insuffisant -> R² avait explosé à -1 488 877)
    "Ridge": RidgeCV(
        alphas=np.logspace(-1, 5, 50)
    ),

    "DecisionTree": DecisionTreeRegressor(
        random_state=42,
        max_depth=4,
        min_samples_leaf=8
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        max_depth=4,
        min_samples_leaf=8,
        max_features="sqrt"
    ),

    # CORRECTIF 2 : régularisation renforcée
    # (écart train/test précédent = 0.672, signe de surapprentissage fort)
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=100,          # 150 -> 100
        learning_rate=0.03,        # 0.05 -> 0.03
        max_depth=2,
        subsample=0.8,             # nouveau : sous-échantillonnage
        min_samples_leaf=10        # nouveau : feuilles moins spécifiques
    )
}

scoring = {
    "R2": "r2",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error"
}

resultats_regression = []

for nom_modele, modele in modeles_regression.items():

    pipeline = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", modele)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y_regression,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    resultats_regression.append({
        "modele": nom_modele,

        "MAE_moyen": round(-scores["test_MAE"].mean(), 4),
        "MAE_ecart_type": round(scores["test_MAE"].std(), 4),

        "RMSE_moyen": round(-scores["test_RMSE"].mean(), 4),
        "RMSE_ecart_type": round(scores["test_RMSE"].std(), 4),

        "R2_moyen": round(scores["test_R2"].mean(), 4),
        "R2_ecart_type": round(scores["test_R2"].std(), 4),

        "R2_train_moyen": round(scores["train_R2"].mean(), 4),
        "R2_train_ecart_type": round(scores["train_R2"].std(), 4),

        "ecart_R2_train_test": round(
            scores["train_R2"].mean() - scores["test_R2"].mean(),
            4
        )
    })

df_resultats_regression = pd.DataFrame(resultats_regression)

df_resultats_regression = df_resultats_regression.sort_values(
    by="RMSE_moyen",
    ascending=True
).reset_index(drop=True)

print("===== Résultats des modèles de régression avec validation croisée =====")
display(df_resultats_regression)

meilleur_modele_regression_nom = df_resultats_regression.iloc[0]["modele"]

print("\nMeilleur modèle selon le RMSE moyen :", meilleur_modele_regression_nom)

PATH_RESULTATS_REGRESSION = OUTPUT_DIR / "01_resultats_modeles_regression_cv_corrige.csv"

df_resultats_regression.to_csv(
    PATH_RESULTATS_REGRESSION,
    index=False,
    encoding="utf-8-sig"
)

print("Fichier exporté :", PATH_RESULTATS_REGRESSION)

===== Résultats des modèles de régression avec validation croisée =====


,modele,MAE_moyen,MAE_ecart_type,RMSE_moyen,RMSE_ecart_type,R2_moyen,R2_ecart_type,R2_train_moyen,R2_train_ecart_type,ecart_R2_train_test
0,RandomForest,19.9593,1.6172,22.9679,1.6344,2.750000e-02,3.030000e-02,0.1743,0.0147,1.469000e-01
1,GradientBoosting,19.4998,1.4159,23.0751,1.2197,1.390000e-02,7.410000e-02,0.4073,0.0227,3.934000e-01
2,DummyRegressor,20.5783,1.6827,23.5198,1.7576,-1.900000e-02,1.790000e-02,0.0000,0.0000,1.900000e-02
3,DecisionTree,21.1885,1.3686,26.1834,1.7433,-2.686000e-01,1.107000e-01,0.3060,0.0531,5.746000e-01
4,Ridge,2181.5429,4322.5198,11040.9844,22035.2527,-1.121149e+06,2.242299e+06,0.2320,0.1090,1.121150e+06



Meilleur modèle selon le RMSE moyen : RandomForest
Fichier exporté : outputs/05_modelisation_predictive/01_resultats_modeles_regression_cv_corrige.csv
